In [1]:
# **Objective:** Explain model predictions using SHAP values
# 
# **Key analyses:**
# 1. Global feature importance
# 2. Local explanations for individual predictions
# 3. Feature interaction effects
# 4. SHAP dependence plots
# 5. Business insights from explanations


# Set up

In [2]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import os
import warnings
import json
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Create output directories FIRST
os.makedirs('../reports/figures', exist_ok=True)
os.makedirs('../models', exist_ok=True)
os.makedirs('../reports', exist_ok=True)


d:\VT\Telecom-Customer-Churn-Prediction-using-Machine-Learning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load data and model


In [3]:

print("=" * 80)
print("LOADING MODEL AND DATA")
print("=" * 80)

X = pd.read_csv('../data/processed/features_scaled.csv')
y = pd.read_csv('../data/processed/target.csv')['Churn']

# Load best model
model_files = [f for f in os.listdir('../models') if f.startswith('best_model_')]
if model_files:
    best_model_path = f"../models/{model_files[0]}"
    if best_model_path.endswith('.pkl'):
        model = joblib.load(best_model_path)
        print(f"Loaded model: {best_model_path}")
        print(f"Model type: {type(model).__name__}")
    else:
        print("Neural network model - SHAP may not work well")
        exit()
else:
    print("No model found! Please run model training notebook first.")
    exit()


LOADING MODEL AND DATA
Loaded model: ../models/best_model_logistic_regression.pkl
Model type: LogisticRegression


# SELECTING APPROPRIATE SHAP EXPLAINER

In [4]:
print("\n" + "=" * 80)
print("SELECTING APPROPRIATE SHAP EXPLAINER")
print("=" * 80)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

model_type = type(model).__name__
print(f"\n Detected Model Type: {model_type}")

# Select appropriate explainer based on model type
if model_type in ['RandomForestClassifier', 'XGBClassifier', 'LGBMClassifier', 
                  'GradientBoostingClassifier', 'ExtraTreesClassifier']:
    print(" Model is tree-based → Using TreeExplainer (FASTEST)")
    explainer = shap.TreeExplainer(model)
    explainer_type = "tree"
    
elif model_type in ['LogisticRegression', 'LinearRegression', 'SGDClassifier']:
    print(" Model is linear → Using LinearExplainer")
    background = X.sample(min(1000, len(X)), random_state=42)
    explainer = shap.LinearExplainer(model, background)
    explainer_type = "linear"
    
elif model_type in ['MLPClassifier', 'MLPRegressor']:
    print(" Model is neural network → Using DeepExplainer")
    background = X.sample(min(100, len(X)), random_state=42)
    explainer = shap.DeepExplainer(model, background)
    explainer_type = "deep"
    
else:
    print(" Model type not recognized → Using KernelExplainer (SLOWEST)")
    background = X.sample(min(50, len(X)), random_state=42)
    explainer = shap.KernelExplainer(model.predict_proba, background)
    explainer_type = "kernel"

print(f"\n SHAP explainer created successfully!")
print(f"   Explainer type: {explainer_type.upper()}")
print("=" * 80)

# Sample data for explanation
print("\n📦 PREPARING DATA SAMPLE")
print("=" * 80)

sample_size = min(1000, len(X))
X_sample = X.sample(n=sample_size, random_state=42)

print(f"Total dataset size: {len(X):,}")
print(f"Sample size for SHAP: {sample_size:,}")
print(f"Features: {X_sample.shape[1]}")

print("\n🔍 SAFETY CHECK: Identifying key columns for segment analysis")
print("-" * 80)

def find_column(df, patterns, default=None):
    """Safely find column matching patterns with fallback"""
    for pattern in patterns:
        matches = [col for col in df.columns if pattern.lower() in col.lower()]
        if matches:
            return matches[0]
    return default

# Identify critical columns with fallbacks
TENURE_COL = find_column(
    X_sample, 
    ['tenure', 'tenure_years', 'tenure_bucket', 'lifecycle'],
    default=X_sample.columns[0]  # Fallback to first column
)

CHARGE_COL = find_column(
    X_sample,
    ['monthly_charge', 'spend', 'revenue', 'total_charge'],
    default=X_sample.columns[1]  # Fallback to second column
)

CONTRACT_COL = find_column(
    X_sample,
    ['contract', 'month_to_month', 'early_termination'],
    default=None
)

print(f"✅ Identified columns:")
print(f"   • Tenure column: '{TENURE_COL}'")
print(f"   • Charge column: '{CHARGE_COL}'")
print(f"   • Contract column: {f'\'{CONTRACT_COL}\'' if CONTRACT_COL else 'Not found'}")
print("=" * 80)

# %%
# Calculate SHAP values
print("\n⏳ CALCULATING SHAP VALUES")
print("=" * 80)
print("   (This may take 1-5 minutes depending on model and sample size)")

if explainer_type == "kernel":
    X_sample_small = X_sample.sample(min(100, len(X_sample)), random_state=42)
    print(f"   Using smaller sample for KernelExplainer: {len(X_sample_small)}")
    shap_values = explainer.shap_values(X_sample_small)
    X_sample = X_sample_small
else:
    shap_values = explainer.shap_values(X_sample)

print(f"✅ SHAP values calculated!")
print(f"   Type: {type(shap_values)}")

# Handle different SHAP output formats
if isinstance(shap_values, list):
    if len(shap_values) == 2:
        shap_values_for_plot = shap_values[1]
        print(f"   Binary classification → Using positive class values")
    else:
        shap_values_for_plot = shap_values[0]
else:
    shap_values_for_plot = shap_values

print(f"   Final shape: {np.array(shap_values_for_plot).shape}")
print("=" * 80)



SELECTING APPROPRIATE SHAP EXPLAINER

 Detected Model Type: LogisticRegression
 Model is linear → Using LinearExplainer

 SHAP explainer created successfully!
   Explainer type: LINEAR

📦 PREPARING DATA SAMPLE
Total dataset size: 7,043
Sample size for SHAP: 1,000
Features: 76

🔍 SAFETY CHECK: Identifying key columns for segment analysis
--------------------------------------------------------------------------------
✅ Identified columns:
   • Tenure column: 'spend_per_tenure'
   • Charge column: 'monthly_charges'
   • Contract column: 'contract_length_score'

⏳ CALCULATING SHAP VALUES
   (This may take 1-5 minutes depending on model and sample size)
✅ SHAP values calculated!
   Type: <class 'numpy.ndarray'>
   Final shape: (1000, 76)



# GLOBAL EXPLANATIONS


In [5]:
print("\n📊 GENERATING GLOBAL EXPLANATIONS")
print("=" * 80)

# 1. Summary plot
print("\n1. Global Feature Importance (Summary Plot)")
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values_for_plot, X_sample, show=False, plot_size=(12, 10))
plt.title('Global Feature Importance', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/shap_summary.png', dpi=300, bbox_inches='tight')
plt.close()
print("   Saved to: ../reports/figures/shap_summary.png")

# 2. Feature importance bar chart
print("\n2. Feature Importance Bar Chart")
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values_for_plot, X_sample, plot_type="bar", show=False, plot_size=(12, 10))
plt.title('Feature Importance (Mean Absolute SHAP)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/shap_importance_bar.png', dpi=300, bbox_inches='tight')
plt.close()
print("   Saved to: ../reports/figures/shap_importance_bar.png")

# 3. SHAP values distribution
print("\n3. SHAP Values Distribution")
shap_df = pd.DataFrame(shap_values_for_plot, columns=X_sample.columns)
mean_abs_shap = shap_df.abs().mean().sort_values(ascending=False)

plt.figure(figsize=(14, 10))
mean_abs_shap.head(20).plot(kind='barh')
plt.title('Top 20 Features by Mean Absolute SHAP', fontsize=16, fontweight='bold')
plt.xlabel('Mean Absolute SHAP Value')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/figures/shap_mean_abs.png', dpi=300, bbox_inches='tight')
plt.close()
print("   Saved to: ../reports/figures/shap_mean_abs.png")



📊 GENERATING GLOBAL EXPLANATIONS

1. Global Feature Importance (Summary Plot)
   Saved to: ../reports/figures/shap_summary.png

2. Feature Importance Bar Chart
   Saved to: ../reports/figures/shap_importance_bar.png

3. SHAP Values Distribution
   Saved to: ../reports/figures/shap_mean_abs.png



# LOCAL EXPLANATIONS

In [6]:
print("\n🔍 GENERATING LOCAL EXPLANATIONS")
print("=" * 80)

# Use shap_values_for_plot for index selection (FIXED)
high_churn_prob_idx = np.argsort(shap_values_for_plot.sum(axis=1))[-3:]
low_churn_prob_idx = np.argsort(shap_values_for_plot.sum(axis=1))[:3]

print(f"\n4. Waterfall plots for high-risk customers (indices: {high_churn_prob_idx})")
for i, idx in enumerate(high_churn_prob_idx):
    try:
        plt.figure(figsize=(10, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values=shap_values_for_plot[idx],  # FIXED: use shap_values_for_plot
                base_values=explainer.expected_value if not isinstance(explainer.expected_value, np.ndarray) else explainer.expected_value[1] if len(explainer.expected_value) > 1 else explainer.expected_value[0],
                data=X_sample.iloc[idx].values,
                feature_names=X_sample.columns.tolist()
            ),
            max_display=10,
            show=False
        )
        plt.title(f'Customer #{idx} - High Churn Risk', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'../reports/figures/shap_waterfall_high_{i}.png', dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   Saved high-risk customer {i+1}")
    except Exception as e:
        print(f"    Skipped high-risk customer {i}: {str(e)}")

print(f"\n5. Waterfall plots for low-risk customers (indices: {low_churn_prob_idx})")
for i, idx in enumerate(low_churn_prob_idx):
    try:
        plt.figure(figsize=(10, 6))
        shap.waterfall_plot(
            shap.Explanation(
                values=shap_values_for_plot[idx],  # FIXED: use shap_values_for_plot
                base_values=explainer.expected_value if not isinstance(explainer.expected_value, np.ndarray) else explainer.expected_value[1] if len(explainer.expected_value) > 1 else explainer.expected_value[0],
                data=X_sample.iloc[idx].values,
                feature_names=X_sample.columns.tolist()
            ),
            max_display=10,
            show=False
        )
        plt.title(f'Customer #{idx} - Low Churn Risk', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'../reports/figures/shap_waterfall_low_{i}.png', dpi=300, bbox_inches='tight')
        plt.close()
        print(f"   Saved low-risk customer {i+1}")
    except Exception as e:
        print(f"    Skipped low-risk customer {i}: {str(e)}")

# FEATURE DEPENDENCE
print("\n GENERATING FEATURE DEPENDENCE PLOTS")
print("=" * 80)

top_features = mean_abs_shap.head(6).index.tolist()
print(f"\n6. SHAP Dependence Plots for top features: {top_features}")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, feature in enumerate(top_features):
    try:
        shap.dependence_plot(
            feature,
            shap_values_for_plot,  
            X_sample,
            ax=axes[i],
            show=False,
            interaction_index='auto'
        )
        axes[i].set_title(f'SHAP Dependence - {feature}', fontsize=12, fontweight='bold')
    except Exception as e:
        axes[i].text(0.5, 0.5, f'Error: {str(e)}', ha='center', va='center')
        axes[i].set_title(f'Error - {feature}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/shap_dependence.png', dpi=300, bbox_inches='tight')
plt.close()
print("    Saved to: ../reports/figures/shap_dependence.png")


# BUSINESS INSIGHTS 
print("\n💡 GENERATING BUSINESS INSIGHTS")
print("=" * 80)

print("\nTop 10 Churn Drivers:")
top_drivers = mean_abs_shap.head(10)
for rank, (feature, importance) in enumerate(top_drivers.items(), 1):
    avg_shap = shap_df[feature].mean()
    direction = "↑ increases" if avg_shap > 0 else "↓ decreases"
    print(f"  {rank}. {feature:30s} | {importance:6.4f} | {direction} churn risk")

# SEGMENT-SPECIFIC INSIGHTS 
print("\n" + "=" * 80)
print("SEGMENT-SPECIFIC INSIGHTS (ROBUST IMPLEMENTATION)")
print("=" * 80)

# HIGH-VALUE CUSTOMERS ANALYSIS
if CHARGE_COL:
    try:
        high_value_mask = X_sample[CHARGE_COL] > X_sample[CHARGE_COL].quantile(0.75)
        if high_value_mask.sum() > 10:  # Minimum sample size
            high_value_shap = pd.DataFrame(
                shap_values_for_plot[high_value_mask],  # FIXED
                columns=X_sample.columns
            )
            high_value_importance = high_value_shap.abs().mean().sort_values(ascending=False).head(5)
            
            print(f"\n High-Value Customers (top 25% by '{CHARGE_COL}'):")
            print("   Top churn drivers:")
            for feature, importance in high_value_importance.items():
                print(f"     • {feature}: {importance:.4f}")
        else:
            print(f"\n High-value segment too small (n={high_value_mask.sum()})")
    except Exception as e:
        print(f"\n High-value analysis skipped: {str(e)}")
else:
    print("\n High-value analysis skipped (charge column not found)")

# NEW CUSTOMERS ANALYSIS
if TENURE_COL:
    try:
        # Smart tenure interpretation
        is_risk_score = any(kw in TENURE_COL.lower() for kw in ['risk', 'score', 'bucket'])
        
        if is_risk_score:
            # Higher value = newer customer (risk score)
            new_customer_mask = X_sample[TENURE_COL] > X_sample[TENURE_COL].quantile(0.75)
            segment_desc = "highest risk score"
        else:
            # Lower value = newer customer (actual tenure)
            new_customer_mask = X_sample[TENURE_COL] < X_sample[TENURE_COL].quantile(0.25)
            segment_desc = "lowest tenure"
        
        if new_customer_mask.sum() > 10:
            new_customer_shap = pd.DataFrame(
                shap_values_for_plot[new_customer_mask],  # FIXED
                columns=X_sample.columns
            )
            new_customer_importance = new_customer_shap.abs().mean().sort_values(ascending=False).head(5)
            
            print(f"\n New Customers ({segment_desc} by '{TENURE_COL}'):")
            print("   Top churn drivers:")
            for feature, importance in new_customer_importance.items():
                print(f"     • {feature}: {importance:.4f}")
        else:
            print(f"\n New customer segment too small (n={new_customer_mask.sum()})")
    except Exception as e:
        print(f"\n New customer analysis skipped: {str(e)}")
else:
    print("\n New customer analysis skipped (tenure column not found)")

# CONTRACT TYPE ANALYSIS (if available)
if CONTRACT_COL and 'month' in CONTRACT_COL.lower():
    try:
        # For one-hot encoded month-to-month column
        mtm_mask = X_sample[CONTRACT_COL] > 0.5
        if mtm_mask.sum() > 10:
            mtm_shap = pd.DataFrame(
                shap_values_for_plot[mtm_mask],  # FIXED
                columns=X_sample.columns
            )
            mtm_importance = mtm_shap.abs().mean().sort_values(ascending=False).head(5)
            
            print(f"\n Month-to-Month Contract Customers ('{CONTRACT_COL}'):")
            print("   Top churn drivers:")
            for feature, importance in mtm_importance.items():
                print(f"     • {feature}: {importance:.4f}")
    except Exception as e:
        print(f"\n  Contract analysis skipped: {str(e)}")



🔍 GENERATING LOCAL EXPLANATIONS

4. Waterfall plots for high-risk customers (indices: [133 936 680])
   Saved high-risk customer 1
   Saved high-risk customer 2
   Saved high-risk customer 3

5. Waterfall plots for low-risk customers (indices: [765  31 644])
   Saved low-risk customer 1
   Saved low-risk customer 2
   Saved low-risk customer 3

 GENERATING FEATURE DEPENDENCE PLOTS

6. SHAP Dependence Plots for top features: ['contract_tenure_interaction', 'internet_service_level', 'total_charges_log', 'contract_length_score', 'avg_monthly_spend', 'monthly_charges']
    Saved to: ../reports/figures/shap_dependence.png

💡 GENERATING BUSINESS INSIGHTS

Top 10 Churn Drivers:
  1. contract_tenure_interaction    | 0.6917 | ↓ decreases churn risk
  2. internet_service_level         | 0.6880 | ↑ increases churn risk
  3. total_charges_log              | 0.6515 | ↓ decreases churn risk
  4. contract_length_score          | 0.5745 | ↑ increases churn risk
  5. avg_monthly_spend              | 0

# SAVE REPORTS 

In [7]:

print("\n" + "=" * 80)
print("SAVING REPORTS AND ARTIFACTS")
print("=" * 80)

# Generate SHAP report DataFrame
shap_report = pd.DataFrame({
    'feature': X_sample.columns,
    'mean_shap': shap_df.mean(),
    'mean_abs_shap': shap_df.abs().mean(),
    'std_shap': shap_df.std(),
    'shap_positive_pct': (shap_df > 0).mean() * 100
})
shap_report = shap_report.sort_values('mean_abs_shap', ascending=False)
shap_report.to_csv('../reports/shap_analysis_report.csv', index=False)
print(" SHAP analysis report saved to ../reports/shap_analysis_report.csv")

# Save metadata about columns used
metadata = {
    'analysis_date': pd.Timestamp.now().isoformat(),
    'model_type': model_type,
    'explainer_type': explainer_type,
    'sample_size': len(X_sample),
    'columns_used': {
        'tenure_column': TENURE_COL,
        'charge_column': CHARGE_COL,
        'contract_column': CONTRACT_COL
    },
    'top_features': mean_abs_shap.head(10).index.tolist()
}

with open('../reports/shap_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(" Metadata saved to ../reports/shap_metadata.json")

# Save SHAP artifacts
import pickle
shap_artifacts = {
    'explainer_type': explainer_type,
    'shap_values': shap_values_for_plot,  # Save the processed version
    'X_sample': X_sample,
    'feature_importance': shap_report,
    'metadata': metadata
}

with open('../models/shap_explainer.pkl', 'wb') as f:
    pickle.dump(shap_artifacts, f)
print(" SHAP artifacts saved to ../models/shap_explainer.pkl")

# %%
print("\n" + "=" * 80)
print(" SHAP EXPLAINABILITY ANALYSIS COMPLETED SUCCESSFULLY!")
print("=" * 80)
print("\n Generated artifacts:")
print("   • SHAP summary plots (../reports/figures/)")
print("   • Feature importance charts")
print("   • Local explanation plots (waterfall)")
print("   • Dependence plots for top features")
print("   • SHAP analysis report (CSV + JSON metadata)")
print("   • SHAP explainer object (pickle)")
print("\n Key improvements in this version:")
print("   •  ROBUST COLUMN HANDLING - No more KeyError!")
print("   •  SMART TENURE INTERPRETATION - Detects risk scores vs actual tenure")
print("   •  ERROR HANDLING - Skips analyses gracefully when columns missing")
print("   •  MINIMUM SAMPLE SIZE CHECKS - Prevents statistical errors")
print("   •  METADATA TRACKING - Documents all column choices")
print("   •  PROPER RESOURCE MANAGEMENT - plt.close() prevents memory leaks")
print("\n" + "=" * 80)


SAVING REPORTS AND ARTIFACTS
 SHAP analysis report saved to ../reports/shap_analysis_report.csv
 Metadata saved to ../reports/shap_metadata.json
 SHAP artifacts saved to ../models/shap_explainer.pkl

 SHAP EXPLAINABILITY ANALYSIS COMPLETED SUCCESSFULLY!

 Generated artifacts:
   • SHAP summary plots (../reports/figures/)
   • Feature importance charts
   • Local explanation plots (waterfall)
   • Dependence plots for top features
   • SHAP analysis report (CSV + JSON metadata)
   • SHAP explainer object (pickle)

 Key improvements in this version:
   •  ROBUST COLUMN HANDLING - No more KeyError!
   •  SMART TENURE INTERPRETATION - Detects risk scores vs actual tenure
   •  ERROR HANDLING - Skips analyses gracefully when columns missing
   •  MINIMUM SAMPLE SIZE CHECKS - Prevents statistical errors
   •  METADATA TRACKING - Documents all column choices
   •  PROPER RESOURCE MANAGEMENT - plt.close() prevents memory leaks

